In [1]:
import os, json, time, pickle
import numpy as np
import torch

os.environ["OMP_NUM_THREADS"] = "4"     
torch.set_num_threads(4)
DEVICE = "cpu"
torch.set_default_dtype(torch.float32)
ROOT   = os.path.abspath(".")
CKPT   = os.path.join(ROOT, "checkpoints")   # models, histories, ground truth
OUT    = os.path.join(ROOT, "outputs", "xai")# XAI figures + manifest
for d in (CKPT, OUT):
    os.makedirs(d, exist_ok=True)
print("checkpoints ->", CKPT)
print("xai outputs ->", OUT)

def save_pickle(obj, name):
    p = os.path.join(CKPT, name)
    with open(p, "wb") as f: pickle.dump(obj, f)
    print(f"  saved {name}")
    return p

def load_pickle(name):
    with open(os.path.join(CKPT, name), "rb") as f:
        return pickle.load(f)

def exists(name):
    return os.path.exists(os.path.join(CKPT, name))

def stamp(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

checkpoints -> d:\PINN\Burgers_1d\checkpoints
xai outputs -> d:\PINN\Burgers_1d\outputs\xai


In [2]:
import importlib
required = ["config", "models", "physics", "train", "train_qapinn",
            "ground_truth", "evaluate", "solver_spectral", "solver_fdm", "solver_fem"]
for m in required:
    importlib.import_module(m)
    print("  ok:", m)

import xai
from xai.adapter import QuantumProbe, TorchModelAdapter
from xai import report, layer2, layer3, domain, scaling
print("  ok: xai", xai.__version__)
os.makedirs("outputs", exist_ok=True)
stamp("Cell 1 done — all imports resolve")

  ok: config
  ok: models
  ok: physics
  ok: train
  ok: train_qapinn
  ok: ground_truth
  ok: evaluate
  ok: solver_spectral
  ok: solver_fdm
  ok: solver_fem
  ok: xai 1.0.0
[18:44:14] Cell 1 done — all imports resolve


In [4]:
from ground_truth import GroundTruth, cross_validate

GT_CACHE = "ground_truth.pkl"
if exists(GT_CACHE):
    stamp("loading cached ground truth")
    gts = load_pickle(GT_CACHE)
else:
    stamp("solving spectral / fdm / fem references (one-time)")
    gts = {m: GroundTruth(m) for m in ("spectral", "fdm", "fem")}
    save_pickle(gts, GT_CACHE)

gt = gts["spectral"]              
import numpy as np
xq = np.linspace(-1, 1, 401)
for t0 in (0.25, 0.5, 0.75):
    ref = gt.slice(t0, xq)
    e_fdm = np.linalg.norm(gts["fdm"].slice(t0, xq) - ref) / np.linalg.norm(ref)
    e_fem = np.linalg.norm(gts["fem"].slice(t0, xq) - ref) / np.linalg.norm(ref)
    print(f"  t={t0}:  FDM vs spectral {e_fdm:.2e}   FEM vs spectral {e_fem:.2e}")

stamp("Cell 2 done — ground truth ready")

[18:45:30] loading cached ground truth
  t=0.25:  FDM vs spectral 6.22e-04   FEM vs spectral 3.04e-04
  t=0.5:  FDM vs spectral 2.17e-03   FEM vs spectral 6.74e-04
  t=0.75:  FDM vs spectral 1.82e-03   FEM vs spectral 4.75e-04
[18:45:30] Cell 2 done — ground truth ready


In [5]:
from models import ClassicalPINN
from train import train_classical
from evaluate import evaluate

CLS_W    = "classical_pinn.pt"
CLS_HIST = "classical_hist.pkl"

CLS_CFG = dict(depth=4, width=8, adam_epochs=600, lbfgs_iter=0,
               lr=2e-3, seed=0, log_every=50,
               snapshot_epochs=(0, 100, 200, 300, 400, 500))

if exists(CLS_W) and exists(CLS_HIST):
    stamp("loading trained classical PINN")
    cl = ClassicalPINN(depth=CLS_CFG["depth"], width=CLS_CFG["width"]).to(DEVICE)
    cl.load_state_dict(torch.load(os.path.join(CKPT, CLS_W), map_location=DEVICE))
    cl.eval()
    cl_hist = load_pickle(CLS_HIST)
else:
    stamp("training classical PINN (this is the long one — grab coffee)")
    t0 = time.perf_counter()
    cl, cl_hist, cl_snaps = train_classical(**CLS_CFG)
    stamp(f"classical training wall = {time.perf_counter()-t0:.1f}s")
    torch.save(cl.state_dict(), os.path.join(CKPT, CLS_W))
    save_pickle(cl_hist, CLS_HIST)
    save_pickle(cl_snaps, "classical_snaps.pkl")

# report accuracy vs spectral GT
res_cl = evaluate(cl, gt, nx=401, nt=101)
print(f"  classical  rel-L2 = {res_cl['l2_global']:.4e}   Linf = {res_cl['linf']:.3e}")
save_pickle(res_cl, "classical_eval.pkl")
stamp("Cell 3 done — classical PINN trained & saved")

[18:46:44] training classical PINN (this is the long one — grab coffee)
[adam ]      0 | 2.8503e+01 | pde 9.07e-01 ic 2.71e-01 bc 1.11e+00 | 1.4s
[adam ]     50 | 9.0637e+00 | pde 3.95e-02 ic 3.96e-01 bc 5.55e-02 | 2.3s
[adam ]    100 | 6.8789e+00 | pde 2.89e-02 ic 2.98e-01 bc 4.45e-02 | 3.2s
[adam ]    150 | 2.1705e+00 | pde 2.83e-01 ic 7.92e-02 bc 1.52e-02 | 4.0s
[adam ]    200 | 1.4361e+00 | pde 5.29e-01 ic 3.50e-02 bc 1.03e-02 | 4.8s
[adam ]    250 | 1.2838e+00 | pde 5.16e-01 ic 2.68e-02 bc 1.16e-02 | 5.7s
[adam ]    300 | 1.1885e+00 | pde 5.04e-01 ic 2.35e-02 bc 1.07e-02 | 6.7s
[adam ]    350 | 1.1153e+00 | pde 4.94e-01 ic 2.14e-02 bc 9.69e-03 | 7.7s
[adam ]    400 | 1.0623e+00 | pde 4.86e-01 ic 1.99e-02 bc 8.95e-03 | 8.5s
[adam ]    450 | 1.0277e+00 | pde 4.81e-01 ic 1.89e-02 bc 8.45e-03 | 9.4s
[adam ]    500 | 1.0084e+00 | pde 4.78e-01 ic 1.83e-02 bc 8.18e-03 | 10.3s
[adam ]    550 | 1.0002e+00 | pde 4.77e-01 ic 1.81e-02 bc 8.06e-03 | 11.1s
[adam ]    599 | 9.9818e-01 | pde 4.76

In [7]:
# CELL 4 — train the QA-PINN from scratch on CPU (the quantum arm)


from models import QAPINN
from train_qapinn import train_qapinn

QA_W    = "qapinn_4q.pt"
QA_HIST = "qapinn_4q_hist.pkl"
QA_SNAP = "qapinn_4q_snaps.pkl"
QA_CFG = dict(n_qubits=3, n_layers=2, hidden=8, reupload=False,
              epochs=600, lr=2e-3, seed=0,
              n_pde=256, n_ic=128, n_bc=128, log_every=50,
              snapshot_epochs=(0, 100, 200, 300, 400, 500))

if exists(QA_W) and exists(QA_HIST) and exists(QA_SNAP):
    stamp("loading trained QA-PINN")
    qa = QAPINN(n_qubits=QA_CFG["n_qubits"], hidden=QA_CFG["hidden"],
                n_layers=QA_CFG["n_layers"], reupload=QA_CFG["reupload"]).to(DEVICE)
    qa.load_state_dict(torch.load(os.path.join(CKPT, QA_W), map_location=DEVICE))
    qa.eval()
    qa_hist  = load_pickle(QA_HIST)
    qa_snaps = load_pickle(QA_SNAP)
else:
    stamp("training QA-PINN (quantum sim on CPU — slowest cell)")
    t0 = time.perf_counter()
    qa, qa_hist, qa_snaps = train_qapinn(**QA_CFG)
    stamp(f"QA-PINN training wall = {time.perf_counter()-t0:.1f}s")
    torch.save(qa.state_dict(), os.path.join(CKPT, QA_W))
    save_pickle(qa_hist,  QA_HIST)
    save_pickle(qa_snaps, QA_SNAP)

res_qa = evaluate(qa, gt, nx=401, nt=101)
print(f"  QA-PINN   rel-L2 = {res_qa['l2_global']:.4e}   Linf = {res_qa['linf']:.3e}")
save_pickle(res_qa, "qapinn_4q_eval.pkl")
stamp("Cell 4 done — QA-PINN trained & saved")

[18:49:31] training QA-PINN (quantum sim on CPU — slowest cell)
[3q L2 ru=0]     0 | 1.0904e+01 | pde 7.58e-05 ic 5.38e-01 bc 7.12e-03 | 0.1s
[3q L2 ru=0]    50 | 9.5499e+00 | pde 1.37e-03 ic 4.54e-01 bc 2.35e-02 | 5.6s
[3q L2 ru=0]   100 | 8.5398e+00 | pde 4.39e-03 ic 3.70e-01 bc 5.72e-02 | 10.9s
[3q L2 ru=0]   150 | 5.7834e+00 | pde 5.20e-02 ic 2.35e-01 bc 5.19e-02 | 17.0s
[3q L2 ru=0]   200 | 3.6641e+00 | pde 1.17e-01 ic 1.17e-01 bc 6.01e-02 | 23.2s
[3q L2 ru=0]   250 | 1.5002e+00 | pde 1.58e-01 ic 4.74e-02 bc 1.96e-02 | 29.1s
[3q L2 ru=0]   300 | 7.3893e-01 | pde 3.50e-01 ic 1.50e-02 bc 4.49e-03 | 34.6s
[3q L2 ru=0]   350 | 6.9301e-01 | pde 4.19e-01 ic 1.05e-02 bc 3.22e-03 | 40.1s
[3q L2 ru=0]   400 | 6.8234e-01 | pde 4.25e-01 ic 9.99e-03 bc 2.88e-03 | 45.7s
[3q L2 ru=0]   450 | 6.7645e-01 | pde 4.26e-01 ic 9.82e-03 bc 2.70e-03 | 51.4s
[3q L2 ru=0]   500 | 6.7343e-01 | pde 4.27e-01 ic 9.74e-03 bc 2.61e-03 | 56.7s
[3q L2 ru=0]   550 | 6.7221e-01 | pde 4.27e-01 ic 9.70e-03 bc 2.57e-0

In [8]:
# CELL 5 — wire trained models into the xai module via adapters
from physics import build_batches, composite_loss
from config import NU, X_MIN, X_MAX, T_MIN, T_MAX

BOUNDS = [(X_MIN, X_MAX), (T_MIN, T_MAX)]    
probe     = QuantumProbe.from_qapinn(qa, device=DEVICE, d_in=2)
classical = TorchModelAdapter(cl, device=DEVICE, name="classical_pinn", d_in=2)

B_fixed = build_batches(n_pde=256, n_ic=128, n_bc=128, seed=123)
loss_q = lambda: composite_loss(qa, B_fixed)[0]    
loss_c = lambda: composite_loss(cl, B_fixed)[0]     


def residual_fn(adapter, X):
    Xr = X.clone().detach().requires_grad_(True)
    x, t = Xr[:, 0:1], Xr[:, 1:2]
    u  = adapter(torch.cat([x, t], 1))
    ut = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    ux = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    uxx= torch.autograd.grad(ux, x, torch.ones_like(ux), create_graph=True)[0]
    return (ut + u*ux - NU*uxx).detach().cpu().numpy()

# sanity: residual should be small on the trained QA-PINN interior
import numpy as np
Xt = torch.rand(200, 2) * torch.tensor([2.0, 1.0]) + torch.tensor([-1.0, 0.0])
print("  mean |PDE residual| (QA-PINN):", np.abs(residual_fn(probe, Xt)).mean())
stamp("Cell 5 done — adapters + closures ready")

  mean |PDE residual| (QA-PINN): 0.65962553
[18:50:58] Cell 5 done — adapters + closures ready


In [9]:
# CELL 6 — LAYER 2: quantum-layer explainability (the core battery)


L2 = {}
stamp("2.1 input sensitivity");        L2["2.1"] = layer2.input_sensitivity(probe, BOUNDS, n=1500, outdir=OUT)
stamp("2.2 measurement operators");    L2["2.2"] = layer2.measurement_operators(probe, BOUNDS, residual_fn=residual_fn, n=800, outdir=OUT)
stamp("2.4 entanglement");             L2["2.4"] = layer2.entanglement_analysis(probe, BOUNDS, residual_fn=residual_fn, n=512, outdir=OUT)
stamp("2.5 fourier spectrum");         L2["2.5"] = layer2.fourier_spectrum(probe, BOUNDS, classical_ref=classical, outdir=OUT)
stamp("2.6 expressivity");             L2["2.6"] = layer2.expressivity_analysis(probe, BOUNDS, n_states=400, outdir=OUT)
stamp("2.9 measurement distribution"); L2["2.9"] = layer2.measurement_distribution(probe, BOUNDS, n=800, outdir=OUT)
stamp("2.10 feature attribution (Q)"); L2["2.10_q"] = layer2.feature_attribution(probe, BOUNDS, outdir=OUT)
stamp("2.10 feature attribution (C)"); L2["2.10_c"] = layer2.feature_attribution(classical, BOUNDS, outdir=OUT)

# 2.8 loss landscape (filter-normalised). n=21 grid = 441 loss evals; a few min.
stamp("2.8 loss landscape (slow-ish)")
L2["2.8"] = layer2.loss_landscape(probe, lambda: float(loss_q()), span=1.0, n=21, outdir=OUT)

# headline numbers
print(f"\n  Meyer–Wallach Q            = {L2['2.4']['meyer_wallach_Q']:.3f}  ({L2['2.4']['interpretation'][:60]}...)")
print(f"  spectral centroid (Q)      = {L2['2.5']['spectral_centroid']:.2f}   reachable f ≤ {L2['2.5']['theoretical_reachable_freq']}")
print(f"  effective dimension        = {L2['2.6']['effective_dimension']:.1f} / {L2['2.6']['feature_dim']}")
print(f"  measurement entropy        = {L2['2.9']['mean_entropy']:.2f} bits")

save_pickle(L2, "layer2_results.pkl")
stamp("Cell 6 done — Layer 2 core complete (see outputs/xai/*.png)")

[18:51:32] 2.1 input sensitivity
[18:51:32] 2.2 measurement operators
[18:51:32] 2.4 entanglement
[18:51:33] 2.5 fourier spectrum
[18:51:34] 2.6 expressivity
[18:51:34] 2.9 measurement distribution
[18:51:34] 2.10 feature attribution (Q)
[18:51:35] 2.10 feature attribution (C)
[18:51:35] 2.8 loss landscape (slow-ish)

  Meyer–Wallach Q            = 0.283  (Mild entanglement: the quantum layer uses correlations modes...)
  spectral centroid (Q)      = 4.19   reachable f ≤ 1
  effective dimension        = 1.7 / 8
  measurement entropy        = 1.57 bits
  saved layer2_results.pkl
[18:52:00] Cell 6 done — Layer 2 core complete (see outputs/xai/*.png)


In [10]:
# CELL 7 — LAYER 3: optimisation geometry (classical vs quantum, side by side)

L3 = {}
stamp("Layer 3 — QA-PINN optimisation report")
L3["qa"] = layer3.optimization_report(probe, loss_q, hist=qa_hist, outdir=OUT)

stamp("Layer 3 — classical optimisation report")
L3["cl"] = layer3.optimization_report(classical, loss_c, hist=cl_hist, outdir=OUT)

for tag, r in (("QA-PINN", L3["qa"]), ("classical", L3["cl"])):
    h = r["hessian"]
    print(f"\n  {tag}:")
    print(f"    grad-norm {r['gradient']['grad_norm_mean']:.3e}  param-norm {r['gradient']['param_norm']:.3e}")
    if h:
        print(f"    λmax {h['lambda_max']:.3e}  cond {h['condition_number']:.2e}  lr-ceiling {h['lr_ceiling']:.2e}")
    if r["stability"]:
        print(f"    90% converged @ iter {r['stability']['iters_to_90pct']}  late-vol {r['stability']['late_volatility']:.2e}")

save_pickle(L3, "layer3_results.pkl")
stamp("Cell 7 done — Layer 3 complete")

[18:53:22] Layer 3 — QA-PINN optimisation report
[18:53:49] Layer 3 — classical optimisation report

  QA-PINN:
    grad-norm 4.345e+00  param-norm 7.521e+00
    λmax 4.589e+03  cond 2.04e+05  lr-ceiling 4.36e-04
    90% converged @ iter 245  late-vol 4.37e-03

  classical:
    grad-norm 5.989e+00  param-norm 6.247e+00
    λmax 4.777e+03  cond 7.53e+05  lr-ceiling 4.19e-04
    90% converged @ iter 130  late-vol 1.79e-02
  saved layer3_results.pkl
[18:53:52] Cell 7 done — Layer 3 complete


In [11]:
# CELL 8 — LAYER 2.11: quantum-state evolution across training

from models import QAPINN

# extract q_weights arrays from the saved state_dict snapshots
weight_snaps = {ep: sd["q_weights"].cpu().numpy() for ep, sd in qa_snaps.items()}
print("  snapshot epochs:", sorted(weight_snaps))

def make_probe_from_weights(w):
    m = QAPINN(n_qubits=QA_CFG["n_qubits"], n_layers=QA_CFG["n_layers"],
               reupload=QA_CFG["reupload"]).to(DEVICE)
    with torch.no_grad():
        m.q_weights.copy_(torch.tensor(w))
    return QuantumProbe.from_qapinn(m, device=DEVICE, d_in=2)

stamp("2.11 quantum-state evolution")
evo = layer2.quantum_state_evolution(make_probe_from_weights, weight_snaps,
                                     BOUNDS, outdir=OUT)
print("  fidelity between consecutive snapshots:", [round(f, 3) for f in evo["fidelity"]])
save_pickle(evo, "layer2_11_evolution.pkl")
stamp("Cell 8 done — state evolution complete")

  snapshot epochs: [0, 100, 200, 300, 400, 500]
[18:53:56] 2.11 quantum-state evolution
  fidelity between consecutive snapshots: [1.0, 0.936, 0.95, 0.976, 1.0, 1.0]
  saved layer2_11_evolution.pkl
[18:53:57] Cell 8 done — state evolution complete


In [12]:
# ============================================================================
# CELL 9 — LAYER 2.7: barren-plateau scan (no training — random inits only)
# ============================================================================
# Measures gradient variance vs #qubits and depth. Each fresh probe is a NEW
# random-init QA-PINN. Keep qubit_list modest on CPU (6 qubits = 64-dim states).

def make_fresh(nq, depth):
    m = QAPINN(n_qubits=nq, n_layers=depth, reupload=False).to(DEVICE)
    return QuantumProbe.from_qapinn(m, device=DEVICE, d_in=2)

stamp("2.7 barren-plateau scan (CPU: keep qubits ≤ 6)")
bp = layer2.barren_plateau_analysis(
    make_fresh, BOUNDS,
    qubit_list=(2, 4, 6), depth_list=(2, 4, 8),
    n_samples=40, n_points=64, outdir=OUT)
save_pickle(bp, "layer2_7_barren.pkl")
stamp("Cell 9 done — barren-plateau scan complete")

[18:54:14] 2.7 barren-plateau scan (CPU: keep qubits ≤ 6)
  saved layer2_7_barren.pkl
[18:54:31] Cell 9 done — barren-plateau scan complete


In [13]:
# ============================================================================
# CELL 10 — LAYER 2.3: circuit-depth sweep (short trainings, checkpointed)
# ============================================================================
# Trains a SMALL QA-PINN at each depth so the sweep is affordable on CPU.
# Each depth's model is cached so re-running is instant.

from train_qapinn import train_qapinn

DEPTHS = (1, 2, 4, 6, 8, 10)        

def build_train_eval(depth):
    tag = f"depth_{depth}.pkl"
    if exists(tag):
        return load_pickle(tag)
    stamp(f"  training depth={depth}")
    m, h, _ = train_qapinn(n_qubits=4, n_layers=depth, reupload=False,
                           epochs=2000, n_pde=256, n_ic=128, n_bc=64,
                           log_every=50, seed=0)
    r = evaluate(m, gt, nx=201, nt=51)
    # gradient norm at the end
    p = QuantumProbe.from_qapinn(m, device=DEVICE, d_in=2)
    gd = layer3.gradient_diagnostics(p, lambda: composite_loss(m, B_fixed)[0], n_batches=3)
    ex = layer2.expressivity_analysis(p, BOUNDS, n_states=200, plot=False, outdir=OUT)
    out = dict(train_loss=float(h["total"][-1]), rel_l2=float(r["l2_global"]),
               grad_norm=float(gd["grad_norm_mean"]),
               expressivity=float(ex["effective_dimension"]),
               runtime=float(h["wall"][-1]))
    save_pickle(out, tag)
    return out

stamp("2.3 circuit-depth sweep")
depth_res = layer2.circuit_depth_analysis(build_train_eval, depths=DEPTHS,
                                          outdir=OUT, name="qapinn_4q")
save_pickle(depth_res, "layer2_3_depth.pkl")
stamp("Cell 10 done — depth sweep complete")

[18:55:37] 2.3 circuit-depth sweep
[18:55:37]   training depth=1
[4q L1 ru=0]     0 | 2.0903e+01 | pde 6.60e-04 ic 6.87e-01 bc 3.58e-01 | 0.1s
[4q L1 ru=0]    50 | 1.0742e+01 | pde 4.72e-04 ic 5.36e-01 bc 7.74e-04 | 5.2s
[4q L1 ru=0]   100 | 1.0695e+01 | pde 4.98e-04 ic 5.35e-01 bc 1.66e-04 | 10.4s
[4q L1 ru=0]   150 | 1.0693e+01 | pde 6.02e-04 ic 5.34e-01 bc 1.37e-04 | 15.8s
[4q L1 ru=0]   200 | 1.0691e+01 | pde 6.96e-04 ic 5.34e-01 bc 1.16e-04 | 21.1s
[4q L1 ru=0]   250 | 1.0690e+01 | pde 7.62e-04 ic 5.34e-01 bc 9.84e-05 | 27.7s
[4q L1 ru=0]   300 | 1.0688e+01 | pde 7.86e-04 ic 5.34e-01 bc 8.25e-05 | 33.7s
[4q L1 ru=0]   350 | 1.0687e+01 | pde 7.68e-04 ic 5.34e-01 bc 6.78e-05 | 39.7s
[4q L1 ru=0]   400 | 1.0686e+01 | pde 7.12e-04 ic 5.34e-01 bc 5.44e-05 | 45.3s
[4q L1 ru=0]   450 | 1.0686e+01 | pde 6.33e-04 ic 5.34e-01 bc 4.24e-05 | 50.8s
[4q L1 ru=0]   500 | 1.0685e+01 | pde 5.45e-04 ic 5.34e-01 bc 3.22e-05 | 56.0s
[4q L1 ru=0]   550 | 1.0685e+01 | pde 4.57e-04 ic 5.34e-01 bc 2.38e-

In [ ]:
# CELL 11 — QUBIT SCALING: {2,4,6} qubits
def build_probe_and_metrics(nq):
    tag = f"scaling_{nq}q.pt"
    htag = f"scaling_{nq}q_hist.pkl"
    if exists(tag) and exists(htag):
        m = QAPINN(n_qubits=nq, n_layers=6, reupload=False).to(DEVICE)
        m.load_state_dict(torch.load(os.path.join(CKPT, tag), map_location=DEVICE))
        h = load_pickle(htag)
    else:
        stamp(f"  training scaling model nq={nq}")
        m, h, _ = train_qapinn(n_qubits=nq, n_layers=6, reupload=False,
                               epochs=2000, n_pde=256, n_ic=128, n_bc=128,
                               log_every=1000, seed=0)
        torch.save(m.state_dict(), os.path.join(CKPT, tag))
        save_pickle(h, htag)
    r = evaluate(m, gt, nx=201, nt=51)
    p = QuantumProbe.from_qapinn(m, device=DEVICE, d_in=2)
    return p, dict(rel_l2=float(r["l2_global"]), runtime=float(h["wall"][-1]))

stamp("qubit-scaling sweep {2,4,6}")
scale_res = scaling.qubit_scaling(build_probe_and_metrics, BOUNDS,
                                  qubit_list=(2, 4, 6, 8), n_eval=300,
                                  outdir=OUT, name="qapinn")
save_pickle(scale_res, "scaling_results.pkl")
print("\n  qubit-scaling table:")
for row in scale_res["table"]:
    print(f"    {row['n_qubits']}q: rel-L2 {row.get('rel_l2', float('nan')):.3e} "
          f"eff-dim {row.get('effective_dimension', float('nan')):.1f} "
          f"Q {row.get('meyer_wallach_Q', float('nan')):.2f} "
          f"H {row.get('measurement_entropy', float('nan')):.2f}")
stamp("Cell 11 done — qubit scaling complete")

[20:40:22] qubit-scaling sweep {2,4,6}
[20:40:22]   training scaling model nq=2
[2q L6 ru=0]     0 | 1.3315e+01 | pde 2.03e-04 ic 5.55e-01 bc 1.10e-01 | 0.1s
[2q L6 ru=0]  1000 | 5.4501e-01 | pde 3.71e-01 ic 6.22e-03 bc 2.50e-03 | 132.1s
[2q L6 ru=0]  1999 | 4.9406e-01 | pde 3.73e-01 ic 4.33e-03 bc 1.71e-03 | 264.0s
  saved scaling_2q_hist.pkl
[20:44:46]   training scaling model nq=4
[4q L6 ru=0]     0 | 1.1045e+01 | pde 1.53e-03 ic 5.45e-01 bc 7.11e-03 | 0.4s
[4q L6 ru=0]  1000 | 4.3770e-01 | pde 3.44e-01 ic 4.39e-03 bc 3.00e-04 | 410.5s
[4q L6 ru=0]  1999 | 4.2379e-01 | pde 3.37e-01 ic 4.13e-03 bc 2.13e-04 | 969.2s
  saved scaling_4q_hist.pkl
[21:00:56]   training scaling model nq=6
[6q L6 ru=0]     0 | 1.0787e+01 | pde 1.10e-06 ic 5.39e-01 bc 1.58e-04 | 1.9s
[6q L6 ru=0]  1000 | 3.4363e-01 | pde 3.04e-01 ic 1.87e-03 bc 9.37e-05 | 1694.2s
[6q L6 ru=0]  1999 | 3.3687e-01 | pde 3.02e-01 ic 1.62e-03 bc 9.95e-05 | 3298.1s
  saved scaling_6q_hist.pkl
[21:55:56]   training scaling model nq

In [ ]:
# ============================================================================
# CELL 12 — LAYER 4: domain generalisation / extrapolation
# ============================================================================
# Extends t beyond the training window [0,1] and measures where each model
# breaks down against the spectral ground truth.
# NOTE: the cached GT was solved to t_max=1.0. To test t>1 honestly, re-solve
# the spectral reference on the extended window first.

from ground_truth import GroundTruth

GT_EXT = "ground_truth_ext.pkl"
if exists(GT_EXT):
    gt_ext = load_pickle(GT_EXT)
else:
    stamp("solving spectral GT on extended window t∈[0,3]")
    gt_ext = GroundTruth("spectral", t_max=3.0)   # solver accepts t_max
    save_pickle(gt_ext, GT_EXT)

ref_fn = lambda P: gt_ext(P[:, 0], P[:, 1])       # (x,t) -> u

stamp("Layer 4 — domain generalisation")
dg = domain.domain_generalization(
    {"qapinn": probe, "classical": classical}, ref_fn, BOUNDS,
    extend_axis=1, factors=(1.0, 1.5, 2.0, 3.0), n=4000, outdir=OUT)
print("  rel-L2 by extension factor:")
for name, errs in dg["rel_l2"].items():
    print(f"    {name}: " + "  ".join(f"{f}×={e:.2e}" for f, e in zip(dg["factors"], errs)))
save_pickle(dg, "layer4_domain.pkl")
stamp("Cell 12 done — Layer 4 complete")